In [1]:
import pandas as pd

In [2]:
inventarios = pd.read_csv('./inventarios.csv')
ventas = pd.read_csv('./ventas.csv')

In [3]:
# @title Parámetros

N_DIAS_QUIEBRE = 2
DIAS_HISTORIAL_VENTAS = 28
PERCENTIL_PRIORIDAD = 0.75

In [4]:
# ============================================================
# Preparación de fechas
# ============================================================

inventarios["fecha"] = pd.to_datetime(inventarios["fecha"])
ventas["fecha"] = pd.to_datetime(ventas["fecha"])

# Fecha de corte simulada: último día disponible en inventarios
fecha_corte = inventarios["fecha"].max()

# Ventana para detectar quiebre actual
inicio_ventana_quiebre = (
    fecha_corte - pd.Timedelta(days=N_DIAS_QUIEBRE - 1)
)

print("Fecha de corte:", fecha_corte.date())
print(
    "Ventana de quiebre:",
    inicio_ventana_quiebre.date(),
    "a",
    fecha_corte.date()
)

# ============================================================
# Detectar candidatos a quiebre
#    Inventario <= 0 durante los últimos N días
# ============================================================

inventario_ultimos_n_dias = inventarios[
    inventarios["fecha"].between(
        inicio_ventana_quiebre,
        fecha_corte
    )
].copy()

inventario_ultimos_n_dias["sin_inventario"] = (
    inventario_ultimos_n_dias["unidades_disponibles"] <= 0
)

# Para cada tienda-producto:
# - debe haber N observaciones
# - todas deben tener inventario <= 0
resumen_inventario = (
    inventario_ultimos_n_dias
    .groupby(
        ["tienda_id", "producto_id"],
        as_index=False
    )
    .agg(
        dias_observados=("fecha", "nunique"),
        dias_sin_inventario=("sin_inventario", "sum"),
        inventario_minimo=("unidades_disponibles", "min"),
        inventario_maximo=("unidades_disponibles", "max")
    )
)

casos_quiebre = resumen_inventario[
    (resumen_inventario["dias_observados"] == N_DIAS_QUIEBRE)
    &
    (resumen_inventario["dias_sin_inventario"] == N_DIAS_QUIEBRE)
].copy()

print("\nCasos de quiebre detectados:", len(casos_quiebre))

# ============================================================
# Definir ventana histórica previa al quiebre
# ============================================================

fin_historial = (
    inicio_ventana_quiebre - pd.Timedelta(days=1)
)

inicio_historial = (
    fin_historial
    - pd.Timedelta(days=DIAS_HISTORIAL_VENTAS - 1)
)

print(
    "Ventana histórica:",
    inicio_historial.date(),
    "a",
    fin_historial.date()
)

# ============================================================
# Obtener ventas históricas
# ============================================================

ventas_historicas = ventas[
    ventas["fecha"].between(
        inicio_historial,
        fin_historial
    )
].copy()

ventas_historicas["importe_venta"] = (
    ventas_historicas["unidades"]
    * ventas_historicas["precio_efectivo"]
)

historico_por_caso = (
    ventas_historicas
    .groupby(
        ["tienda_id", "producto_id"],
        as_index=False
    )
    .agg(
        unidades_28d=("unidades", "sum"),
        ventas_28d_mxn=("importe_venta", "sum")
    )
)

# ============================================================
# Cruzar candidatos de quiebre con su histórico de ventas
# ============================================================

casos_quiebre = casos_quiebre.merge(
    historico_por_caso,
    on=["tienda_id", "producto_id"],
    how="left"
)

# Si un producto no tuvo ventas en la ventana histórica,
# interpretamos el acumulado como 0
casos_quiebre[
    ["unidades_28d", "ventas_28d_mxn"]
] = casos_quiebre[
    ["unidades_28d", "ventas_28d_mxn"]
].fillna(0)

# ============================================================
# Calcular umbral de prioridad con P75
# ============================================================

umbral_unidades = casos_quiebre[
    "unidades_28d"
].quantile(PERCENTIL_PRIORIDAD)

print(
    f"\nPercentil {PERCENTIL_PRIORIDAD:.0%}:",
    umbral_unidades,
    "unidades"
)

# ============================================================
# Seleccionar quiebres prioritarios
# ============================================================

casos_prioritarios = casos_quiebre[
    casos_quiebre["unidades_28d"] >= umbral_unidades
].copy()

porcentaje_prioritarios = (
    len(casos_prioritarios)
    / len(casos_quiebre)
    * 100
    if len(casos_quiebre) > 0
    else 0
)

print("Casos prioritarios:", len(casos_prioritarios))
print(
    f"Porcentaje prioritario: "
    f"{porcentaje_prioritarios:.2f}%"
)


Fecha de corte: 2026-08-30
Ventana de quiebre: 2026-08-29 a 2026-08-30

Casos de quiebre detectados: 1600
Ventana histórica: 2026-08-01 a 2026-08-28

Percentil 75%: 103.0 unidades
Casos prioritarios: 403
Porcentaje prioritario: 25.19%


## Resumen Estadístico

In [5]:
display(
    casos_quiebre[
        ["unidades_28d", "ventas_28d_mxn"]
    ].describe()
)


,unidades_28d,ventas_28d_mxn
count,1600.000000,1600.000000
mean,75.592500,4742.698881
std,48.578378,4518.371988
min,4.000000,35.000000
25%,38.000000,1341.600000
50%,66.000000,3200.015000
75%,103.000000,6854.887500
max,286.000000,28800.000000


In [6]:
# Casos prioritarios ordenados por mayor demanda histórica
display(
    casos_prioritarios[
        [
            "tienda_id",
            "producto_id",
            "dias_observados",
            "dias_sin_inventario",
            "unidades_28d",
            "ventas_28d_mxn"
        ]
    ]
    .sort_values(
        "unidades_28d",
        ascending=False
    )
    .head(20)
)

,tienda_id,producto_id,dias_observados,dias_sin_inventario,unidades_28d,ventas_28d_mxn
594,T0028,P012,2,2,286,11062.48
869,T0040,P056,2,2,269,23099.03
545,T0026,P012,2,2,262,10134.16
274,T0013,P060,2,2,256,28800.00
264,T0013,P028,2,2,255,9524.25
252,T0012,P044,2,2,248,23453.36
605,T0028,P040,2,2,246,17453.70
863,T0040,P035,2,2,244,19873.80
561,T0026,P044,2,2,239,22602.23
601,T0028,P034,2,2,235,19596.65


In [7]:
casos_prioritarios.describe()

,dias_observados,dias_sin_inventario,inventario_minimo,inventario_maximo,unidades_28d,ventas_28d_mxn
count,403.0,403.0,403.0,403.0,403.000000,403.000000
mean,2.0,2.0,0.0,0.0,143.848635,10095.022457
std,0.0,0.0,0.0,0.0,33.614650,5058.335818
min,2.0,2.0,0.0,0.0,103.000000,663.320000
25%,2.0,2.0,0.0,0.0,119.000000,6138.795000
50%,2.0,2.0,0.0,0.0,135.000000,10358.700000
75%,2.0,2.0,0.0,0.0,159.500000,13223.980000
max,2.0,2.0,0.0,0.0,286.000000,28800.000000


In [8]:
print(f"""
  DIAS_MINIMOS_QUIEBRE  = {N_DIAS_QUIEBRE}
  VENTANA_QUIEBRES_DIAS = {DIAS_HISTORIAL_VENTAS}
  UMBRAL_COSTO_MINIMO   = {casos_prioritarios['ventas_28d_mxn'].min():.2f}
  """
)



  DIAS_MINIMOS_QUIEBRE  = 2
  VENTANA_QUIEBRES_DIAS = 28
  UMBRAL_COSTO_MINIMO   = 663.32
  
